# This notebook belongs to the analysis of the 2+1 encoder models after fully finetuned with the best variant (B). 

---

# some required cells from last notebook

## Audit: Dev Scale Runs (ids, manual Macro F1 vs history, TF-IDF on same ids)

In [6]:
# ---- Section: Imports and constants ----
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from digikala_review_transformer import config

RUNS_ROOT = config.ROOT / "artifacts" / "transformer_runs"
RUN_NAMES = ["tooka_B", "tooka_C","mdeberta_B","roberta_B"]
PACK_DIR = config.ROOT / "data" / "training_pack"
BASE_DIR = config.ROOT / "reports" / "predictions"

HEAD_COLUMNS = {
    "rec": ["p_recommended", "p_not_recommended", "p_no_idea"],
    "band": ["p_low", "p_medium", "p_high"],
}
TOL = 1e-5      # tolerance for "same as history.csv"
N_BOOT = 300    # paired bootstrap resamples (kept small on purpose)

n_fail = 0


# ---- Section: Helpers ----
def check(ok: bool, pass_msg: str, fail_msg: str) -> None:
    """Print [PASS] or [FAIL] <reason> -> <fix> and count failures."""
    global n_fail
    if ok:
        print(f"[PASS] {pass_msg}")
    else:
        n_fail += 1
        print(f"[FAIL] {fail_msg}")


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Independent Macro F1 over the 3 classes (sklearn, not the project code)."""
    return float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0))


def valid_mask(series: pd.Series) -> np.ndarray:
    """True where the label is one of 0/1/2 (drops NaN / missing codes)."""
    return series.isin([0, 1, 2]).to_numpy()


# ---- Section: Pack (read without the body column) ----
pack_val = pd.read_parquet(
    PACK_DIR / "validation.parquet",
    columns=["id", "y_rec", "y_band", "y_rec_raw", "y_band_raw"],
)
pack_val["id"] = pack_val["id"].astype(str)
pack_val = pack_val.set_index("id")
pack_test_ids = set(pd.read_parquet(PACK_DIR / "test.parquet", columns=["id"])["id"].astype(str))
print(f"pack validation rows: {len(pack_val):,}\n")

# TF-IDF validation predictions (corrected labels + predicted class)
tfidf = {}
for head in HEAD_COLUMNS:
    frame = pd.read_parquet(
        BASE_DIR / f"tfidf12_{head}_B__validation.parquet",
        columns=["id", "y_true_corr", "pred"],
    )
    frame["id"] = frame["id"].astype(str)
    tfidf[head] = frame.drop_duplicates("id").set_index("id")

rng = np.random.default_rng(2026)

# ---- Section: Audit per run ----
for run_name in RUN_NAMES:
    run_dir = RUNS_ROOT / run_name
    print("=" * 78)
    print(f"RUN: {run_name}")
    print("=" * 78)

    history = pd.read_csv(run_dir / "history.csv")
    best_row = history[history["is_best"]].iloc[-1]
    last_row = history.iloc[-1]
    best_epoch, last_epoch = int(best_row["epoch"]), int(last_row["epoch"])
    print(f"history: best epoch = {best_epoch}, last epoch = {last_epoch}, "
          f"best val_mean = {best_row['val_mean_macro_f1']:.6f}, "
          f"last val_mean = {last_row['val_mean_macro_f1']:.6f}")

    # -- Check 1: ids --
    probs = pd.read_parquet(run_dir / "val_probs_best.parquet")
    probs["id"] = probs["id"].astype(str)
    check(probs["id"].is_unique, f"val ids are unique (n={len(probs):,})",
          "duplicate ids in val_probs_best -> bug in predict/evaluate ordering")
    in_pack = probs["id"].isin(pack_val.index)
    check(bool(in_pack.all()), "all val ids exist in pack validation.parquet",
          f"{int((~in_pack).sum())} ids not in validation split -> wrong data reached the model")
    test_ids = pd.read_parquet(run_dir / "test_probs.parquet", columns=["id"])["id"].astype(str)
    check(test_ids.is_unique and test_ids.isin(pack_test_ids).all(),
          f"test ids are unique and all exist in pack test.parquet (n={len(test_ids):,})",
          "test ids duplicated or not in test split -> check predict.finalize_run")

    labels = pack_val.reindex(probs["id"])
    corrected, raw, tf_cmp = {}, {}, {}

    for head, cols in HEAD_COLUMNS.items():
        pred = probs[cols].to_numpy().argmax(axis=1)

        # -- Check 2: manual Macro F1, corrected view (vs history best row) --
        y_corr = labels[f"y_{head}"]
        m = valid_mask(y_corr)
        corrected[head] = macro_f1(y_corr.to_numpy()[m], pred[m])
        hist_val = float(best_row[f"val_macro_f1_{head}"])
        check(abs(corrected[head] - hist_val) < TOL,
              f"{head} corrected F1 manual={corrected[head]:.6f} == history epoch {best_epoch} ({hist_val:.6f}), rows={int(m.sum()):,}",
              f"{head} corrected F1 manual={corrected[head]:.6f} != history {hist_val:.6f} -> wrong ids/labels/order or wrong checkpoint saved")

        # -- Check 3: raw view (history column found by name, never guessed blindly) --
        y_raw = labels[f"y_{head}_raw"]
        mr = valid_mask(y_raw)
        raw[head] = macro_f1(y_raw.to_numpy()[mr], pred[mr])
        cand = [c for c in history.columns if "raw" in c and f"f1_{head}" in c]
        if len(cand) == 1:
            hist_raw = float(best_row[cand[0]])
            check(abs(raw[head] - hist_raw) < TOL,
                  f"{head} raw F1 manual={raw[head]:.6f} == history {cand[0]} ({hist_raw:.6f})",
                  f"{head} raw F1 manual={raw[head]:.6f} != history {hist_raw:.6f} -> raw view computed from wrong labels")
        else:
            print(f"[SKIP] {head} raw: history columns found = {cand}; manual raw F1 = {raw[head]:.6f}")

        # -- Same-ids comparison with TF-IDF --
        tf = tfidf[head].reindex(probs["id"])
        ok_rows = m & tf["pred"].notna().to_numpy() & valid_mask(tf["y_true_corr"])
        both = m & valid_mask(tf["y_true_corr"])
        agree = float((tf["y_true_corr"].to_numpy()[both] == y_corr.to_numpy()[both]).mean())
        check(agree == 1.0, f"{head}: pack labels == TF-IDF file labels on shared ids",
              f"{head}: label agreement with TF-IDF file = {agree:.4f} -> label view mismatch between pack and baseline")
        yt = y_corr.to_numpy()[ok_rows]
        pt = pred[ok_rows]
        pb = tf["pred"].to_numpy()[ok_rows].astype(int)
        f_t, f_b = macro_f1(yt, pt), macro_f1(yt, pb)
        diffs = np.empty(N_BOOT)
        for i in range(N_BOOT):
            idx = rng.integers(0, len(yt), len(yt))
            diffs[i] = macro_f1(yt[idx], pt[idx]) - macro_f1(yt[idx], pb[idx])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        tf_cmp[head] = (len(yt), f_t, f_b, f_t - f_b, lo, hi)

    # -- Check 2b: mean + best-vs-last distinction --
    mean_manual = (corrected["rec"] + corrected["band"]) / 2
    check(abs(mean_manual - float(best_row["val_mean_macro_f1"])) < TOL,
          f"val_mean manual={mean_manual:.6f} == history best row ({float(best_row['val_mean_macro_f1']):.6f})",
          "val_mean manual != history best row -> selection metric computed differently")
    if best_epoch != last_epoch:
        check(abs(mean_manual - float(last_row["val_mean_macro_f1"])) > TOL,
              f"probabilities come from best epoch {best_epoch}, not last epoch {last_epoch}",
              "saved probabilities equal the LAST epoch -> best checkpoint/probs not saved from best epoch")

    # -- Report tables --
    print("\nManual Macro F1 on this run's validation subset:")
    print(f"  corrected: rec={corrected['rec']:.4f}  band={corrected['band']:.4f}")
    print(f"  raw      : rec={raw['rec']:.4f}  band={raw['band']:.4f}")
    print("\nTransformer vs TF-IDF on the SAME ids (corrected labels, paired bootstrap 95% CI of the difference):")
    for head, (n, f_t, f_b, d, lo, hi) in tf_cmp.items():
        print(f"  {head:<5} n={n:,}  transformer={f_t:.4f}  tfidf={f_b:.4f}  diff={d:+.4f}  CI=[{lo:+.4f}, {hi:+.4f}]")
    print()

# ---- Section: Final verdict ----
print("=" * 78)
if n_fail == 0:
    print("AUDIT RESULT: ALL CHECKS PASSED")
else:
    print(f"AUDIT RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

pack validation rows: 922,794

RUN: tooka_B
history: best epoch = 4, last epoch = 4, best val_mean = 0.663465, last val_mean = 0.663465
[PASS] val ids are unique (n=922,794)
[PASS] all val ids exist in pack validation.parquet
[PASS] test ids are unique and all exist in pack test.parquet (n=922,805)
[PASS] rec corrected F1 manual=0.720016 == history epoch 4 (0.720016), rows=788,677
[PASS] rec raw F1 manual=0.717183 == history val_macro_f1_rec_raw (0.717183)
[PASS] rec: pack labels == TF-IDF file labels on shared ids
[PASS] band corrected F1 manual=0.606914 == history epoch 4 (0.606914), rows=922,794
[PASS] band raw F1 manual=0.598873 == history val_macro_f1_band_raw (0.598873)
[PASS] band: pack labels == TF-IDF file labels on shared ids
[PASS] val_mean manual=0.663465 == history best row (0.663465)

Manual Macro F1 on this run's validation subset:
  corrected: rec=0.7200  band=0.6069
  raw      : rec=0.7172  band=0.5989

Transformer vs TF-IDF on the SAME ids (corrected labels, paired bo

---

# Step 1: Check the integrity of the received files (read-only)

In [ ]:
# ============================================================================
# Inspect 15: integrity of the four runs (read-only; nothing is written)
# Needs cell 1 (RUNS_ROOT, RUN_NAMES, check) in this kernel.
# ============================================================================
import hashlib
import json
import re
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from digikala_review_transformer import config

n_fail = 0
REQUIRED_FILES = [
    "run_summary.json",
    "history.csv",
    "train_log.csv",
    "val_probs_best.parquet",
    "test_probs.parquet",
    "best_model.pt",
]
EXPECTED_ROWS = {
    "val_probs_best.parquet": 922_794,
    "test_probs.parquet": 922_805,
}
HASH_CANDIDATES = [
    Path(RUNS_ROOT)
    / "hashes.txt",  # Dynamically points to artifacts/transformer_runs/hashes.txt
    config.ROOT / "outputs_to_send" / "weights" / "hashes.txt",
    config.ROOT / "outputs_to_send" / "hashes.txt",
    config.ROOT / "hashes.txt",
    config.ROOT / "artifacts" / "hashes.txt",
]
CHUNK = 64 * 1024 * 1024


def resolve_file(run_dir: Path, run_name: str, base_name: str) -> Path:
    """Dynamically resolves '{run_name}_{base_name}' if it exists, otherwise '{base_name}'."""
    prefixed_path = run_dir / f"{run_name}_{base_name}"
    if prefixed_path.is_file():
        return prefixed_path
    return run_dir / base_name


# ---- Section: Files, status, epochs ----
for name in RUN_NAMES:
    run_dir = Path(RUNS_ROOT) / name
    check(
        run_dir.is_dir(),
        f"{name}: folder exists",
        f"{run_dir} is missing -> copy the run folder here",
    )
    if not run_dir.is_dir():
        continue

    resolved_paths = {
        req: resolve_file(run_dir, name, req) for req in REQUIRED_FILES
    }

    absent = [req for req, path in resolved_paths.items() if not path.is_file()]
    check(
        not absent,
        f"{name}: all {len(REQUIRED_FILES)} required files present",
        f"{name}: missing {absent} -> ask for these files again",
    )

    valid_file_names = {path.name for path in resolved_paths.values()}
    extra_files = sorted(
        p.name for p in run_dir.iterdir() if p.name not in valid_file_names
    )
    print(f"   {name}: extra files -> {extra_files}")

    if absent:
        continue

    summary = json.loads(
        resolved_paths["run_summary.json"].read_text(encoding="utf-8")
    )
    check(
        summary.get("status") == "complete",
        f"{name}: status = complete",
        f"{name}: status = {summary.get('status')!r} -> the run did not finish normally",
    )

    hist = pd.read_csv(resolved_paths["history.csv"])
    best = int(hist.loc[hist["val_mean_macro_f1"].idxmax(), "epoch"])
    print(
        f"   {name}: epochs in history = {len(hist)}, best epoch (argmax val mean F1) = {best}, "
        f"best val mean F1 = {hist['val_mean_macro_f1'].max():.6f}"
    )
    print(
        f"   {name}: selection_and_stopping = "
        f"{json.dumps(summary.get('selection_and_stopping', 'key not found'), default=str)[:500]}"
    )

    for fname, expected in EXPECTED_ROWS.items():
        actual_path = resolved_paths[fname]
        got = pq.ParquetFile(actual_path).metadata.num_rows
        check(
            got == expected,
            f"{name}: {actual_path.name} has {got:,} rows",
            f"{name}: {actual_path.name} has {got:,} rows, expected {expected:,} -> wrong or damaged file",
        )


# ---- Section: SHA256 against hashes.txt ----
hash_path = next((p for p in HASH_CANDIDATES if p.is_file()), None)
if hash_path is None:
    found = [
        str(p.relative_to(config.ROOT))
        for p in config.ROOT.rglob("hashes*.txt")
        if ".venv" not in p.parts
    ][:5]
    check(
        False,
        "",
        f"hashes.txt not found in the usual places; search found {found} -> "
        "set HASH_CANDIDATES to its real path",
    )
else:
    raw = hash_path.read_bytes()
    text = (
        raw.decode("utf-16")
        if raw[:2] in (b"\xff\xfe", b"\xfe\xff")
        else raw.decode("utf-8-sig")
    )
    expected_hash = {}
    for line in text.splitlines():
        digest = re.search(r"\b[0-9A-Fa-f]{64}\b", line)
        files = re.findall(r"[^\s\\/]+\.pt\b", line)
        if digest and files:
            # Map the filename (e.g. 'mdeberta_B_best_model.pt') directly to its lowercase hash
            expected_hash[files[-1]] = (
                digest.group(0).lower(),
                line.strip(),
            )

    print(
        f"\nhashes file: {hash_path.relative_to(config.ROOT)} ({len(expected_hash)} parsed lines)"
    )
    for fname, (digest, line) in expected_hash.items():
        print(f"   {digest[:12]}...  {fname}")
    check(
        len(expected_hash) >= len(RUN_NAMES),
        f"hashes file has at least {len(RUN_NAMES)} weight entries",
        f"only {len(expected_hash)} entries parsed -> read the lines above; the format may differ",
    )

    for name in RUN_NAMES:
        run_dir = Path(RUNS_ROOT) / name
        pt = resolve_file(run_dir, name, "best_model.pt")
        if not pt.is_file():
            continue

        # Look up by actual file name (e.g. 'mdeberta_B_best_model.pt') or fallback to prefixed name
        match = expected_hash.get(pt.name) or expected_hash.get(
            f"{name}_best_model.pt"
        )
        if not match:
            print(
                f"   {name}: no matching line in hashes file for {pt.name} (skipped)"
            )
            continue

        expected_digest, _ = match
        sha = hashlib.sha256()
        with pt.open("rb") as handle:
            while chunk := handle.read(CHUNK):
                sha.update(chunk)
        check(
            sha.hexdigest() == expected_digest,
            f"{name}: {pt.name} SHA256 matches ({pt.stat().st_size / 1e6:.0f} MB)",
            f"{name}: SHA256 differs from hashes.txt -> the copy is damaged; copy this file again",
        )


print("=" * 78)
print(
    "INSPECT 15 RESULT: all runs intact"
    if n_fail == 0
    else "INSPECT 15 RESULT: see [FAIL] above"
)

[PASS] tooka_B: folder exists
[PASS] tooka_B: all 6 required files present
   tooka_B: extra files -> ['config_resolved.yaml', 'encoder_config.json', 'tokenizer', 'tooka_B.zip', 'train.log']
[PASS] tooka_B: status = complete
   tooka_B: epochs in history = 4, best epoch (argmax val mean F1) = 4, best val mean F1 = 0.663465
   tooka_B: selection_and_stopping = {"criterion": "val_mean_macro_f1_corrected", "labels_used_for_selection": "step1_corrected_only", "test_used_for_any_decision": false, "max_epochs": 4, "early_stopping_patience": 1, "early_stopping_min_delta": 0.0, "shared_across_heads": true, "epochs_completed": 4, "global_steps_completed": 89384, "best_epoch": 4, "best_val_mean_macro_f1": 0.6634647612355236, "stop_reason": "max_epochs_reached", "best_epoch_history": {"epoch": 4, "train_objective": 1.4917308678988066, "val_mean_macro_f1": 0.663
[PASS] tooka_B: val_probs_best.parquet has 922,794 rows
[PASS] tooka_B: test_probs.parquet has 922,805 rows
[PASS] tooka_C: folder exists

# Stage 1 / Cell 1: Mode switch, paths, parameters and run audit

In [9]:
# ============================================================================
# Stage 1 / Cell 1: Mode switch, paths, parameters and run audit
# Change ONLY `STATE` below. Every later cell reads the variables built here.
# ============================================================================

# ---- Section: THE SWITCH ----
STATE = (
    1  # 0 = dev results (small subset)   |   1 = real results (friend's full runs)
)


# ---- Section: Imports ----
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import pack as pack_module

# ---- Section: Mode table (the only place where 0/1 matters) ----
_STD = {
    "model.max_length": 128,
    "model.tokenizer_backend": "hf_fast",
    "training.batch_size": 128,
}
_MDEBERTA = {
    "model.max_length": 160,
    "model.tokenizer_backend": "sentencepiece",
    "training.batch_size": 64,
}
CONFIG_DEFAULTS = {
    "model.tokenizer_backend": "hf_fast"
}  # key may be absent from config_resolved.yaml (default)

MODES = {
    0: {
        "name": "dev",
        "tag": (
            "DEV (100k-row training subset, 10k-row validation/test subsets)"
        ),
        "runs_root": "artifacts/transformer_runs_dev_scale",
        "run_names": ["tooka_B", "tooka_C"],
        "out_dir": "reports/transformer_stage1/dev",
        "full_data": False,
        # dotted-path -> expected value in config_resolved.yaml
        "expected_config": {"seed": config.SPLIT_SEED},
    },
    1: {
        "name": "full",
        "tag": "FULL (all train_supervised rows, full validation/test)",
        "runs_root": "artifacts/transformer_runs",
        "run_names": ["tooka_B", "tooka_C", "roberta_B", "mdeberta_B"],
        "out_dir": "reports/transformer_stage2/full",
        "full_data": True,
        "expected_config": {
            "seed": config.SPLIT_SEED,
            "training.max_epochs": 4,
            "training.early_stopping_patience": 1,
            "training.lr_encoder": 2e-5,
            "training.lr_heads": 1e-4,
            "training.weight_decay": 0.01,
            "training.warmup_ratio": 0.06,
            "training.label_smoothing": 0.1,
            "training.loss_weight_band": 1.0,
            "training.use_class_weights": True,
            "training.bf16": True,
        },
        "run_overrides": {
            "tooka_B": _STD,
            "tooka_C": _STD,
            "roberta_B": _STD,
            "mdeberta_B": (
                _MDEBERTA
            ),  # batch 64: documented exception (VRAM spill at 128 in smoke)
        },
    },
}
if STATE not in MODES:
    raise ValueError(f"STATE must be 0 or 1, got {STATE!r}")
MODE = MODES[STATE]


# ---- Section: Paths derived from the mode ----
RUNS_ROOT = config.ROOT / MODE["runs_root"]
RUN_NAMES = MODE["run_names"]
OUT_DIR = config.ROOT / MODE["out_dir"]
TABLES_DIR = OUT_DIR / "tables"
FIGS_DIR = OUT_DIR / "figures"
for _d in (TABLES_DIR, FIGS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


PACK_DIR = config.ROOT / "data" / "training_pack"
BASELINE_DIR = config.ROOT / "reports" / "predictions"
BASELINE_IDS = {
    "rec": "tfidf12_rec_B",
    "band": "tfidf12_band_B",
}  # files: <id>__<split>.parquet


# ---- Section: Analysis parameters (same in both modes; fixed by the plan) ----
HEADS = ("rec", "band")
VIEWS = ("raw", "corrected")  # both are reported, neither is "main"
SELECTION_METRIC = "val_mean_macro_f1"  # mean of the two heads' Macro F1 (corrected validation)
C_OVER_B_THRESHOLD = 0.003  # C wins only if score_C >= score_B + 0.003
N_BOOT = 1000  # paired bootstrap resamples (used by later cells)
BOOT_SEED = config.SPLIT_SEED
FIG_DPI = 150
# Aurora Glass palette (same hex values as plot_style): raw = SLATE_LIGHT, corrected = BLUE (as in the baseline phase)
VIEW_COLORS = {"raw": "#94A3B8", "corrected": "#38BDF8"}
VARIANT_COLORS = {"B": "#2DD4BF", "C": "#FBBF24"}  # TURQUOISE / AMBER
MODE_TAG = MODE["tag"]  # put this in every chart caption / table title


REQUIRED_FILES = [
    "config_resolved.yaml",
    "history.csv",
    "train_log.csv",
    "train.log",
    "run_summary.json",
    "val_probs_best.parquet",
    "test_probs.parquet",
    "encoder_config.json",
]
HISTORY_COLUMNS = [
    "epoch",
    "val_mean_macro_f1",
    "val_macro_f1_rec",
    "val_macro_f1_band",
    "epoch_seconds",
    "is_best",
]
TOL = 1e-6


# ---- Section: Helpers ----
n_fail = 0


def check(ok: bool, pass_msg: str, fail_msg: str) -> bool:
    """Print [PASS] or [FAIL] <reason> -> <fix>; count failures."""
    global n_fail
    if ok:
        print(f"[PASS] {pass_msg}")
    else:
        n_fail += 1
        print(f"[FAIL] {fail_msg}")
    return ok


def flatten(obj, prefix: str = "") -> dict:
    """Flatten nested dicts into {'a.b.c': value}."""
    flat = {}
    for key, value in obj.items():
        path = f"{prefix}{key}"
        if isinstance(value, dict):
            flat.update(flatten(value, path + "."))
        else:
            flat[path] = value
    return flat


def same_value(actual, expected) -> bool:
    """Numeric-tolerant equality (YAML may load 2e-05 as a string)."""
    if isinstance(expected, bool):
        return (
            actual is expected or str(actual).lower() == str(expected).lower()
        )
    if isinstance(expected, (int, float)):
        try:
            return abs(float(actual) - float(expected)) < 1e-12
        except (TypeError, ValueError):
            return False
    return actual == expected


def dig(obj: dict, *keys):
    """Nested lookup with a clear error that lists the keys that do exist."""
    cur = obj
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            have = (
                list(cur.keys())
                if isinstance(cur, dict)
                else type(cur).__name__
            )
            raise KeyError(
                f"key '{'.'.join(keys)}' not found; available at that level: {have}"
            )
        cur = cur[key]
    return cur


def parquet_rows(path: Path) -> int:
    return pq.ParquetFile(path).metadata.num_rows


# ---- Section: Pack facts ----
manifest = json.loads(
    (PACK_DIR / pack_module.MANIFEST_NAME).read_text(encoding="utf-8")
)
PACK_FINGERPRINT = pack_module.pack_fingerprint(manifest)
PACK_ROWS = {
    split: manifest["files"][pack_module.pack_file_name(split)]["rows"]
    for split in ("train", "validation", "test")
}


print("=" * 78)
print(f"MODE: STATE={STATE} -> {MODE_TAG}")
print(f"runs root : {RUNS_ROOT.relative_to(config.ROOT)}")
print(f"outputs   : {OUT_DIR.relative_to(config.ROOT)}")
print(f"pack      : fingerprint={PACK_FINGERPRINT}, rows={PACK_ROWS}")
print("=" * 78)


# ---- Section: Audit and load each run ----
RUNS = {}
for run_name in RUN_NAMES:
    print(f"\n--- {run_name} ---")
    run_dir = RUNS_ROOT / run_name
    if not check(
        run_dir.is_dir(),
        f"run directory exists: {run_dir.relative_to(config.ROOT)}",
        f"missing run directory {run_dir} -> "
        f"{'extract the friend zip into artifacts/transformer_runs/' if STATE == 1 else 'run the dev cell first'}",
    ):
        continue

    # -- files --
    missing = [
        f
        for f in REQUIRED_FILES
        if not (run_dir / f).is_file() or (run_dir / f).stat().st_size == 0
    ]
    check(
        not missing,
        "all required files exist and are non-empty",
        f"missing/empty files: {missing} -> re-package the run or re-run training",
    )
    tok_dir = run_dir / "tokenizer"
    check(
        tok_dir.is_dir() and any(tok_dir.iterdir()),
        "tokenizer/ is non-empty",
        "tokenizer/ missing or empty -> re-package the run",
    )
    check(
        not (run_dir / "last.pt").exists(),
        "no last.pt (run was finalized)",
        "last.pt still exists -> the run did not finish cleanly",
    )
    _has_pt = any(
        (run_dir / n).exists()
        for n in ("best_model.pt", f"{run_name}_best_model.pt")
    )
    print(
        f"[INFO] best weights file present: {_has_pt} (not needed by the analysis)"
    )
    if missing:
        continue

    # -- load --
    history = pd.read_csv(run_dir / "history.csv")
    summary = json.loads(
        (run_dir / "run_summary.json").read_text(encoding="utf-8")
    )
    cfg = yaml.safe_load(
        (run_dir / "config_resolved.yaml").read_text(encoding="utf-8")
    )
    flat_cfg = flatten(cfg)

    # -- config --
    check(
        cfg.get("run_name") == run_name
        and cfg["training"]["variant"] == run_name.split("_")[-1],
        f"config run_name/variant match the folder ({run_name}, variant {cfg['training']['variant']})",
        f"config says run_name={cfg.get('run_name')}, variant={cfg['training']['variant']} -> wrong folder/zip",
    )
    expected_cfg = {
        **MODE["expected_config"],
        **MODE.get("run_overrides", {}).get(run_name, {}),
    }
    actual_cfg = {
        k: flat_cfg.get(k, CONFIG_DEFAULTS.get(k)) for k in expected_cfg
    }
    wrong = {
        k: (actual_cfg[k], v)
        for k, v in expected_cfg.items()
        if not same_value(actual_cfg[k], v)
    }
    check(
        not wrong,
        f"config matches the expected {MODE['name']} settings ({len(expected_cfg)} keys)",
        f"config differs from the plan (key: (actual, expected)): {wrong} -> run was made with other settings",
    )

    # -- history --
    miss_cols = [c for c in HISTORY_COLUMNS if c not in history.columns]
    if not check(
        not miss_cols,
        f"history.csv has the required columns ({len(history)} epochs)",
        f"history.csv lacks columns {miss_cols} -> check training/loop.py history writer",
    ):
        continue
    best_row = history[history["is_best"]].iloc[-1]
    check(
        best_row[SELECTION_METRIC] >= history[SELECTION_METRIC].max() - TOL,
        f"is_best marks the maximum {SELECTION_METRIC} (epoch {int(best_row['epoch'])})",
        f"is_best row is not the maximum of {SELECTION_METRIC} -> best-epoch selection bug",
    )
    max_epochs = int(cfg["training"]["max_epochs"])
    if len(history) < max_epochs:
        check(
            not bool(history.iloc[-1]["is_best"]),
            f"stopped early after {len(history)}/{max_epochs} epochs with a non-best last epoch",
            "stopped early although the last epoch was best -> early-stopping logic bug",
        )

    # -- summary keys (from the other-AI digest; reported clearly if missing) --
    try:
        s_best_epoch = int(dig(summary, "selection_and_stopping", "best_epoch"))
        s_best_score = float(
            dig(summary, "selection_and_stopping", "best_val_mean_macro_f1")
        )
        stop_reason = dig(summary, "selection_and_stopping", "stop_reason")
        s_fingerprint = dig(summary, "pack", "fingerprint")
        check(
            s_best_epoch == int(best_row["epoch"])
            and abs(s_best_score - float(best_row[SELECTION_METRIC])) < TOL,
            f"run_summary best epoch/score agree with history (epoch {s_best_epoch}, {s_best_score:.6f})",
            f"summary says epoch {s_best_epoch}/{s_best_score:.6f}, history says "
            f"epoch {int(best_row['epoch'])}/{float(best_row[SELECTION_METRIC]):.6f} -> inconsistent run files",
        )
        check(
            s_fingerprint == PACK_FINGERPRINT,
            f"pack fingerprint matches ({s_fingerprint})",
            f"run pack fingerprint {s_fingerprint} != current pack {PACK_FINGERPRINT} -> trained on a different pack",
        )
        dig(summary, "validation_best")
        dig(summary, "test_quick")
    except KeyError as exc:
        check(
            False,
            "",
            f"run_summary.json key problem: {exc} -> send this line, I will fix the key names",
        )
        continue

    # -- probability file rows --
    n_val = parquet_rows(run_dir / "val_probs_best.parquet")
    n_test = parquet_rows(run_dir / "test_probs.parquet")
    if MODE["full_data"]:
        check(
            n_val == PACK_ROWS["validation"] and n_test == PACK_ROWS["test"],
            f"probability files cover the full splits (val {n_val:,}, test {n_test:,})",
            f"rows val {n_val:,}/test {n_test:,} != pack {PACK_ROWS['validation']:,}/{PACK_ROWS['test']:,} "
            "-> a subset run was packaged as a real run",
        )
    else:
        check(
            n_val < PACK_ROWS["validation"] and n_test < PACK_ROWS["test"],
            f"dev probability files are subsets (val {n_val:,}, test {n_test:,})",
            "dev run covers the full split -> STATE/runs_root mismatch",
        )

    RUNS[run_name] = {
        "dir": run_dir,
        "cfg": cfg,
        "summary": summary,
        "history": history,
        "variant": cfg["training"]["variant"],
        "best_epoch": s_best_epoch,
        "best_score": s_best_score,
        "stop_reason": stop_reason,
        "n_val": n_val,
        "n_test": n_test,
    }


# ---- Section: B vs C must differ only in variant and run_name ----
PAIR = ("tooka_B", "tooka_C")
if all(name in RUNS for name in PAIR):
    flat_a, flat_b = (flatten(RUNS[n]["cfg"]) for n in PAIR)
    diff_keys = {
        k
        for k in set(flat_a) | set(flat_b)
        if flat_a.get(k) != flat_b.get(k)
    }
    check(
        diff_keys == {"training.variant", "run_name"},
        "tooka_B and tooka_C configs differ only in training.variant and run_name",
        f"B/C configs differ in {sorted(diff_keys)} -> unfair comparison, runs must share all settings",
    )

# ---- Section: all runs evaluated on the same rows ----
if RUNS:
    check(
        len({r["n_val"] for r in RUNS.values()}) == 1
        and len({r["n_test"] for r in RUNS.values()}) == 1,
        f"all {len(RUNS)} runs evaluated on the same number of validation/test rows",
        "runs have different row counts -> different evaluation sets",
    )

# ---- Section: Stage-2 settings audit (training section must match tooka_B except the documented items) ----
REFERENCE_RUN = "tooka_B"
RUNTIME_ONLY = {
    "training.num_workers",
    "training.log_every_steps",
    "training.eval_batch_size",
}
EXPECTED_TRAINING_DIFF = {
    "tooka_C": {"training.variant"},
    "roberta_B": set(),
    "mdeberta_B": {"training.batch_size"},
}
if STATE == 1 and all(name in RUNS for name in RUN_NAMES):
    ref_flat = flatten(RUNS[REFERENCE_RUN]["cfg"])
    for name in RUN_NAMES:
        if name == REFERENCE_RUN:
            continue
        cur_flat = flatten(RUNS[name]["cfg"])
        keys = {
            k
            for k in set(ref_flat) | set(cur_flat)
            if ref_flat.get(k) != cur_flat.get(k)
        }
        train_diff = {
            k
            for k in keys
            if (k.startswith("training.") or k == "seed")
            and k not in RUNTIME_ONLY
        }
        check(
            train_diff == EXPECTED_TRAINING_DIFF[name],
            f"{name}: training settings and seed equal {REFERENCE_RUN} except {sorted(EXPECTED_TRAINING_DIFF[name]) or 'nothing'}",
            f"{name}: training differs from {REFERENCE_RUN} in {sorted(train_diff)}, expected "
            f"{sorted(EXPECTED_TRAINING_DIFF[name])} -> resolve or document before comparing",
        )
        print(
            f"   [INFO] {name}: other differing keys vs {REFERENCE_RUN} (model.*, paths, runtime): "
            f"{sorted(keys - train_diff)}"
        )


# ---- Section: Compact overview ----
print("\n" + "=" * 78)
overview = pd.DataFrame(
    [
        {
            "run": n,
            "variant": r["variant"],
            "epochs_done": len(r["history"]),
            "best_epoch": r["best_epoch"],
            "best_val_mean_macro_f1": round(r["best_score"], 6),
            "stop_reason": r["stop_reason"],
        }
        for n, r in RUNS.items()
    ]
)
print(overview.to_string(index=False) if len(overview) else "(no runs loaded)")


ANALYSIS_READY = (n_fail == 0) and len(RUNS) == len(RUN_NAMES)
print("=" * 78)
print(
    "CELL 1 RESULT: ALL CHECKS PASSED"
    if ANALYSIS_READY
    else f"CELL 1 RESULT: {n_fail} CHECK(S) FAILED -> fix before running the next cells"
)

MODE: STATE=1 -> FULL (all train_supervised rows, full validation/test)
runs root : artifacts/transformer_runs
outputs   : reports/transformer_stage2/full
pack      : fingerprint=9faa2a97e4502701, rows={'train': 2860236, 'validation': 922794, 'test': 922805}

--- tooka_B ---
[PASS] run directory exists: artifacts/transformer_runs/tooka_B
[PASS] all required files exist and are non-empty
[PASS] tokenizer/ is non-empty
[PASS] no last.pt (run was finalized)
[INFO] best weights file present: True (not needed by the analysis)
[PASS] config run_name/variant match the folder (tooka_B, variant B)
[PASS] config matches the expected full settings (14 keys)
[PASS] history.csv has the required columns (4 epochs)
[PASS] is_best marks the maximum val_mean_macro_f1 (epoch 4)
[PASS] run_summary best epoch/score agree with history (epoch 4, 0.663465)
[PASS] pack fingerprint matches (9faa2a97e4502701)
[PASS] probability files cover the full splits (val 922,794, test 922,805)

--- tooka_C ---
[PASS] run 

# Stage 2 / Cell 2: Integrity of the probability files (read-only; nothing is written)

In [10]:
# ============================================================================
# Stage 2 / Cell 2: Integrity of the probability files (read-only; nothing is written)
# Needs cell 1 in this kernel (ANALYSIS_READY, RUNS, RUN_NAMES, PACK_DIR, check).
# ============================================================================

# ---- Section: Imports and guards ----
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")
for _name in ("RUNS", "RUN_NAMES", "PACK_DIR", "check"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cell 1 first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
SPLIT_FILES = {"validation": ("val_probs_best.parquet", "validation.parquet"),
               "test": ("test_probs.parquet", "test.parquet")}
GROUP_CLASSES = {"rec": ["recommended", "not_recommended", "no_idea"],
                 "band": ["low", "medium", "high"]}
PROB_COLUMNS = {head: [f"p_{c}" for c in classes] for head, classes in GROUP_CLASSES.items()}
SUM_TOL = 1e-4

# ---- Section: Reference id order from the data pack ----
PACK_IDS = {}
for split, (_, pack_file) in SPLIT_FILES.items():
    PACK_IDS[split] = pq.read_table(PACK_DIR / pack_file, columns=["id"]).to_pandas()["id"].astype(str)
    print(f"pack {split}: {len(PACK_IDS[split]):,} ids, unique = {PACK_IDS[split].is_unique}")

# ---- Section: Checks per run and split ----
for name in RUN_NAMES:
    for split, (prob_file, _) in SPLIT_FILES.items():
        probs = pd.read_parquet(RUNS[name]["dir"] / prob_file)
        absent = [c for head in PROB_COLUMNS.values() for c in head if c not in probs.columns]
        if not check(not absent and "id" in probs.columns,
                     f"{name}/{split}: id and all 6 probability columns present",
                     f"{name}/{split}: missing {absent}; actual columns: {list(probs.columns)} "
                     "-> fix PROB_COLUMNS / GROUP_CLASSES"):
            continue
        ids = probs["id"].astype(str).reset_index(drop=True)
        check(ids.is_unique, f"{name}/{split}: ids are unique ({len(ids):,})",
              f"{name}/{split}: duplicate ids -> damaged file")
        check(len(ids) == len(PACK_IDS[split]) and ids.equals(PACK_IDS[split]),
              f"{name}/{split}: ids identical and in the same order as the pack file",
              f"{name}/{split}: ids differ from the pack in content or order -> "
              "a later cell that aligns by position would be wrong")
        values = probs[[c for head in PROB_COLUMNS.values() for c in head]].to_numpy(dtype=float)
        check(np.isfinite(values).all() and (values >= 0).all() and (values <= 1 + SUM_TOL).all(),
              f"{name}/{split}: all probabilities finite and within [0, 1]",
              f"{name}/{split}: NaN, inf or out-of-range probabilities -> damaged file")
        for head, cols in PROB_COLUMNS.items():
            gap = float(np.abs(probs[cols].to_numpy(dtype=float).sum(axis=1) - 1.0).max())
            check(gap < SUM_TOL, f"{name}/{split}/{head}: row sums equal 1 (max gap {gap:.1e})",
                  f"{name}/{split}/{head}: row sums deviate by {gap:.1e} -> probabilities are not softmax outputs")

print("=" * 78)
print("STAGE 2 / CELL 2 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"STAGE 2 / CELL 2 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

pack validation: 922,794 ids, unique = True
pack test: 922,805 ids, unique = True
[PASS] tooka_B/validation: id and all 6 probability columns present
[PASS] tooka_B/validation: ids are unique (922,794)
[PASS] tooka_B/validation: ids identical and in the same order as the pack file
[PASS] tooka_B/validation: all probabilities finite and within [0, 1]
[PASS] tooka_B/validation/rec: row sums equal 1 (max gap 1.4e-07)
[PASS] tooka_B/validation/band: row sums equal 1 (max gap 1.3e-07)
[PASS] tooka_B/test: id and all 6 probability columns present
[PASS] tooka_B/test: ids are unique (922,805)
[PASS] tooka_B/test: ids identical and in the same order as the pack file
[PASS] tooka_B/test: all probabilities finite and within [0, 1]
[PASS] tooka_B/test/rec: row sums equal 1 (max gap 1.4e-07)
[PASS] tooka_B/test/band: row sums equal 1 (max gap 1.3e-07)
[PASS] tooka_C/validation: id and all 6 probability columns present
[PASS] tooka_C/validation: ids are unique (922,794)
[PASS] tooka_C/validation: i